In [3]:
import pandas as pd
import numpy as np
from pgmpy.estimators import HillClimbSearch, BayesianEstimator, BDeu, BIC, K2
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import BayesianEstimator, MaximumLikelihoodEstimator
from pgmpy.models import DiscreteBayesianNetwork
from sklearn.model_selection import KFold

df = pd.read_csv("consumers_profile.csv")  # datos discretizados

# ── Configuraciones a comparar ──────────────────────────────────────────────
configs = {
    "BDeu_ess1":   BDeu(df, equivalent_sample_size=1),
    "BDeu_ess5":   BDeu(df, equivalent_sample_size=5),
    "BDeu_ess10":  BDeu(df, equivalent_sample_size=10),
    "BDeu_ess50":  BDeu(df, equivalent_sample_size=50),
    "BDeu_ess100": BDeu(df, equivalent_sample_size=100),
    "BIC":         BIC(df),
    "K2":          K2(df),
}

# ── Evaluación con k-fold ────────────────────────────────────────────────────
def log_likelihood(model, data):
    """Log-verosimilitud media por muestra en un conjunto de datos."""
    model.fit(data, estimator=BayesianEstimator, prior_type="BDeu", equivalent_sample_size=10)
    ll = 0
    for _, row in data.iterrows():
        try:
            p = model.get_state_probability(row.to_dict())
            ll += np.log(p + 1e-10)
        except:
            ll += np.log(1e-10)
    return ll / len(data)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = {name: [] for name in configs}

for fold, (train_idx, test_idx) in enumerate(kf.split(df)):
    train, test = df.iloc[train_idx], df.iloc[test_idx]

    for name, scoring_fn in configs.items():
        score_cls = type(scoring_fn)
        score_train = score_cls(train, **({} if name == "K2" else
                                {"equivalent_sample_size": int(name.split("ess")[-1])}
                                if "BDeu" in name else {}))

        hc = HillClimbSearch(train)
        best_dag = hc.estimate(scoring_method=score_train, max_indegree=4)
        model = DiscreteBayesianNetwork(best_dag.edges())

        ll = log_likelihood(model, test)
        results[name].append(ll)
        print(f"[Fold {fold+1}] {name}: LL={ll:.4f}")

# ── Tabla comparativa ────────────────────────────────────────────────────────
rows = []
for name, lls in results.items():
    fold_scores = {f"Fold {i+1}": round(v, 4) for i, v in enumerate(lls)}
    rows.append({
        "Config": name,
        **fold_scores,
        "Media": round(np.mean(lls), 4),
        "Std":   round(np.std(lls), 4),
    })

summary = pd.DataFrame(rows).set_index("Config")
best = summary["Media"].idxmax()
summary = summary.sort_values("Media", ascending=False)
summary.style \
    .highlight_max(subset=["Media"], color="lightgreen") \
    .highlight_min(subset=["Media"], color="salmon") \
    .format(precision=4) \
    .set_caption(f"Resultados k-fold — mejor configuración: {best}")


INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'UserAge': 'C', 'UserGender': 'C', 'HouseholdType': 'C', 'TimeOfDay': 'C', 'DayType': 'C', 'ProgramType': 'C', 'ProgramGenre': 'C', 'ProgramDuration': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'UserAge': 'C', 'UserGender': 'C', 'HouseholdType': 'C', 'TimeOfDay': 'C', 'DayType': 'C', 'ProgramType': 'C', 'ProgramGenre': 'C', 'ProgramDuration': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'UserAge': 'C', 'UserGender': 'C', 'HouseholdType': 'C', 'TimeOfDay': 'C', 'DayType': 'C', 'ProgramType': 'C', 'ProgramGenre': 'C', 'ProgramDuration': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'UserAge': 'C', 'UserGender': 'C', 'HouseholdType': 'C', 'TimeOfDay': 'C', 'DayType': 'C', 'ProgramType': 'C

  0%|          | 0/1000000 [00:00<?, ?it/s]

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'UserAge': 'C', 'UserGender': 'C', 'HouseholdType': 'C', 'TimeOfDay': 'C', 'DayType': 'C', 'ProgramType': 'C', 'ProgramGenre': 'C', 'ProgramDuration': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'UserAge': 'C', 'UserGender': 'C', 'HouseholdType': 'C', 'TimeOfDay': 'C', 'DayType': 'C', 'ProgramType': 'C', 'ProgramGenre': 'C', 'ProgramDuration': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'UserAge': 'C', 'UserGender': 'C', 'HouseholdType': 'C', 'TimeOfDay': 'C', 'DayType': 'C', 'ProgramType': 'C', 'ProgramGenre': 'C', 'ProgramDuration': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'UserAge': 'C', 'UserGender': 'C', 'HouseholdType': 'C', 'TimeOfDay': 'C', 'DayType': 'C', 'ProgramType': 'C

[Fold 1] BDeu_ess1: LL=-6.3146


  0%|          | 0/1000000 [00:00<?, ?it/s]

...

[Fold 5] BIC: LL=-6.2744


  0%|          | 0/1000000 [00:00<?, ?it/s]

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'UserAge': 'C', 'UserGender': 'C', 'HouseholdType': 'C', 'TimeOfDay': 'C', 'DayType': 'C', 'ProgramType': 'C', 'ProgramGenre': 'C', 'ProgramDuration': 'C'}


[Fold 5] K2: LL=-6.2420


,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Media,Std
Config,,,,,,,
K2,-6.2808,-6.2786,-6.2713,-6.2767,-6.2420,-6.2699,0.0143
BDeu_ess100,-6.2950,-6.2998,-6.2861,-6.2885,-6.2604,-6.2860,0.0137
BDeu_ess50,-6.3000,-6.2998,-6.2861,-6.2956,-6.2696,-6.2902,0.0115
BDeu_ess10,-6.3014,-6.3017,-6.2945,-6.2973,-6.2689,-6.2928,0.0122
BDeu_ess5,-6.3031,-6.3028,-6.2964,-6.2989,-6.2744,-6.2951,0.0106
BIC,-6.3099,-6.3095,-6.3032,-6.3020,-6.2744,-6.2998,0.0131
BDeu_ess1,-6.3146,-6.3095,-6.3032,-6.3056,-6.2707,-6.3007,0.0155
